# 10 · Chicago no lugar do FMI

O FMI publica a média do mês com uns 20 dias de atraso. Quero ver se a média dos fechamentos diários de Chicago, que sai na hora, pode tomar o lugar dele no mês que ainda não saiu.

In [ ]:
import sys, json
from pathlib import Path

if Path("/kaggle/input").exists():
    RAIZ = next(Path("/kaggle/input").rglob("src/soja/__init__.py")).parents[2]
else:
    RAIZ = Path.cwd().parent
sys.path.insert(0, str(RAIZ / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from soja import config, dados

ETAPA = "10_chicago_vs_fmi"
PASTA = config.pasta_resultados(ETAPA)
plt.rcParams.update({"figure.figsize": (11, 3.5), "axes.grid": True, "grid.alpha": .3})

fmi = dados.baixar_preco()
diario = dados.baixar_soja_diaria()
diario.index = pd.to_datetime(diario.index).tz_localize(None)
chicago = (diario / 100 * config.BUSHELS_POR_T).groupby(diario.index.to_period("M")).mean().rename("chicago")
print(f"FMI até {fmi.index[-1]}, Chicago até {diario.index[-1]:%d/%m/%Y}")

## 1. Média de Chicago × FMI

In [ ]:
comp = pd.concat([fmi.rename("fmi"), chicago], axis=1).dropna()
comp["dif_log"] = np.log(comp["fmi"]) - np.log(comp["chicago"])
dlog = np.log(comp[["fmi", "chicago"]]).diff().dropna()

print(f"{len(comp)} meses ({comp.index[0]} a {comp.index[-1]})")
print(f"Correlação dos preços: {comp['fmi'].corr(comp['chicago']):.4f}")
print(f"Correlação das variações mensais: {dlog['fmi'].corr(dlog['chicago']):.4f}")
print(f"FMI acima de Chicago, em média: {comp['dif_log'].mean():+.2%}")
print(f"Diferença absoluta média: {comp['dif_log'].abs().mean():.2%}")
for lim in (0.01, 0.02):
    print(f"Meses com diferença até {lim:.0%}: {(comp['dif_log'].abs() <= lim).mean():.0%}")

fig, ax = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
x = comp.index.to_timestamp()
ax[0].plot(x, comp["fmi"], "k", lw=1, label="FMI"); ax[0].plot(x, comp["chicago"], "tab:green", lw=1, label="média de Chicago")
ax[0].set_ylabel("US$/t"); ax[0].legend()
ax[1].bar(x, comp["dif_log"] * 100, width=25, color="gray"); ax[1].set_ylabel("FMI − Chicago (%)")
plt.tight_layout(); plt.savefig(PASTA / "chicago_vs_fmi.png", dpi=120); plt.show()

## 2. Corrigindo pela diferença recente

Se o FMI fica sempre um pouco acima ou abaixo de Chicago, dá para corrigir com a diferença dos 12 meses anteriores (só com o que já se sabia).

In [ ]:
comp["ajuste"] = comp["dif_log"].rolling(12).median().shift(1)
comp["estimado"] = comp["chicago"] * np.exp(comp["ajuste"])
comp["erro_sem"] = comp["dif_log"]
comp["erro_com"] = np.log(comp["fmi"]) - np.log(comp["estimado"])

teste = comp.loc[config.TESTE_INICIO:].dropna()
erro_modelo = pd.read_csv(config.ler_resultado("07_modelos", "resumo.csv")).query("modelo == 'ridge' and periodo == 'teste'")["mae"].iloc[0]
resumo = pd.DataFrame({
    "erro médio absoluto": [teste["erro_sem"].abs().mean(), teste["erro_com"].abs().mean(), erro_modelo],
    "pior mês": [teste["erro_sem"].abs().max(), teste["erro_com"].abs().max(), np.nan],
}, index=["Chicago puro", "Chicago corrigido", "erro do modelo (etapa 07)"])
print(f"Teste: {teste.index[0]} a {teste.index[-1]}")
print(resumo.map(lambda v: f"{v:.2%}" if pd.notna(v) else "-"))

plt.hist(teste["erro_sem"] * 100, bins=30, alpha=.5, label="Chicago puro")
plt.hist(teste["erro_com"] * 100, bins=30, alpha=.5, label="Chicago corrigido")
plt.xlabel("erro em relação ao FMI (%)"); plt.legend(); plt.show()

## 3. Meses que o FMI ainda não publicou

Aqui uso a média de Chicago corrigida para os meses que faltam e comparo com a previsão da etapa 08.

In [ ]:
ultimo_ajuste = comp["dif_log"].iloc[-12:].median()
faltam = chicago.loc[fmi.index[-1] + 1:]
estimados = (faltam * np.exp(ultimo_ajuste)).rename("estimado")
print(f"Correção usada: {ultimo_ajuste:+.2%}")
print(estimados.round(1).to_string())
if diario.index[-1].to_period("M") == estimados.index[-1]:
    print(f"({estimados.index[-1]} ainda está em andamento: média até {diario.index[-1]:%d/%m})")

prev = json.loads(config.ler_resultado("08_modelo_final", "previsao.json").read_text())
mes = pd.Period(prev["mes_previsto"], "M")
if mes in estimados.index:
    real = estimados[mes]
    a, b = prev["faixas"]["80%"]
    print(f"\n{mes}: previsto US$ {prev['preco_previsto']:.1f}, estimado por Chicago US$ {real:.1f} "
          f"(erro {np.log(real / prev['preco_previsto']):+.2%}); dentro da faixa de 80%: {a <= real <= b}")
    hist = fmi.loc[mes - 12:]
    plt.plot(hist.index.to_timestamp(), hist, "k-o", ms=3, label="FMI")
    plt.plot(estimados.index.to_timestamp(), estimados, "o", c="tab:green", label="estimado por Chicago")
    plt.errorbar(mes.to_timestamp(), prev["preco_previsto"], yerr=[[prev["preco_previsto"] - a], [b - prev["preco_previsto"]]],
                 fmt="s", c="tab:red", capsize=4, label="previsão (faixa 80%)")
    plt.ylabel("US$/t"); plt.legend(); plt.show()

## 4. Salvando

In [ ]:
comp.to_csv(PASTA / "comparacao.csv", index_label="mes")
estimados.to_csv(PASTA / "estimados.csv", index_label="mes")
config.salvar_config(
    PASTA,
    meses=[str(comp.index[0]), str(comp.index[-1])],
    correlacao_precos=float(comp["fmi"].corr(comp["chicago"])),
    correlacao_variacoes=float(dlog["fmi"].corr(dlog["chicago"])),
    erro_medio_teste={"chicago_puro": float(teste["erro_sem"].abs().mean()), "chicago_corrigido": float(teste["erro_com"].abs().mean())},
    correcao_atual=float(ultimo_ajuste),
    fmi_ate=str(fmi.index[-1]), chicago_ate=str(diario.index[-1].date()),
)
print(sorted(p.name for p in PASTA.iterdir()))